In [10]:
from pathlib import Path
import pandas as pd
import folium
from IPython.display import display, IFrame

# =========================================================
# Paths
# =========================================================

DATA_PATH = Path("../data/real/locations.csv")

OUT_DIR = Path("../results/maps")
OUT_DIR.mkdir(parents=True, exist_ok=True)

HTML_PATH = OUT_DIR / "folium_locations_map.html"

# =========================================================
# Read data
# =========================================================

locations = pd.read_csv(DATA_PATH)

# =========================================================
# Create map
# =========================================================

m = folium.Map(
    tiles="CartoDB positron"
)

# auto fit all cafes nicely
m.fit_bounds([
    [locations["latitude"].min(), locations["longitude"].min()],
    [locations["latitude"].max(), locations["longitude"].max()]
])

# =========================================================
# Add locations
# =========================================================

for _, row in locations.iterrows():

    lat = row["latitude"]
    lon = row["longitude"]
    name = row["name"]

    if str(row["is_depot"]).lower() == "yes":

        folium.Marker(
            location=[lat, lon],
            tooltip=name,
            popup=f"<b>{name}</b><br>Depot",
            icon=folium.Icon(
                color="red",
                icon="star",
                prefix="fa"
            )
        ).add_to(m)

    else:

        folium.CircleMarker(
            location=[lat, lon],
            radius=5,
            color="blue",
            fill=True,
            fill_opacity=0.85,
            tooltip=name,
            popup=f"<b>{name}</b>"
        ).add_to(m)

# =========================================================
# Save html
# =========================================================

m.save(HTML_PATH)

print(f"HTML saved to: {HTML_PATH}")

# =========================================================
# Display in notebook
# =========================================================

display(
    IFrame(
        str(HTML_PATH),
        width=1000,
        height=700
    )
)

# =========================================================
# Export high-quality PNG
# =========================================================

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
import time

PNG_PATH = OUT_DIR / "folium_locations_map.png"

options = Options()

options.add_argument("--headless")

# good balance between quality and size
options.add_argument("--window-size=1400,1000")

# retina quality
options.add_argument("--force-device-scale-factor=2")

driver = webdriver.Chrome(options=options)

driver.set_window_size(1000, 800)

# open local html
driver.get(f"file://{HTML_PATH.resolve()}")

# wait for tiles to fully load
time.sleep(4)

# save screenshot
driver.save_screenshot(str(PNG_PATH))

driver.quit()

print(f"PNG saved to: {PNG_PATH}")

HTML saved to: ..\results\maps\folium_locations_map.html


PNG saved to: ..\results\maps\folium_locations_map.png
